## Ćwiczenie 1

Utwórz macierz M o wymiarach 1000x1000 o elementach A[i,j] == 2 * i - 1 * j. 

In [ ]:
import numpy as np
n = 1000
# za pomocą list comprehension
M = np.array([[2*i - j for j in range(1000)] for i in range(1000)])
# za pomocą wektorów 
i = np.arange(0, n, 1).reshape(n, 1) # array range
j = np.arange(0, n, 1).reshape(1, n)
# broadcasting tutaj - warto zastosować 
M = 2*i - j
# M.shape , M
# czy można to zrobić szybciej 

## Ćwiczenie 2
Oblicz `a+b`, raz dostosowując macierz `a`, a raz dostosowując macierz `b`.
Wypisz `shape` dla macierzy `a`, `b`, `a+b`.

<code>
a = np.array([[  1.,   2.,   3.],
              [11.,  12.,  13.],
              [21.,  22.,  23.],
              [31.,  32.,  33.]])

b = np.array([1.0, 2.0, 3.0, 4.0])
</code>

In [ ]:
a = np.array([[  1.,   2.,   3.],
              [11.,  12.,  13.],
              [21.,  22.,  23.],
              [31.,  32.,  33.]])

b = np.array([1.0, 2.0, 3.0, 4.0])

a. shape, b.shape
a+b.reshape(4, -1), a.T + b


## Ćwiczenie 3

Proszę "ręcznie" zwektoryzować poniższy kod:

<code>
def test_equality(x):
    if x**2 + 2*x-1 == (x + 2)*x-1:
        return True
    else:
        return False
</code>

<code>
test_equality(np.linspace(0,1,1000))
</code>

In [ ]:
# wyjściowa funkcja
def test_equality(x):
    # czy chodzi o to, że rozpatruje osobno dwa metody przez sprawdzenie typu obiektu czy chodzi o to żebym dodał 
    return x**2 + 2*x-1 == (x + 2)*x-1

# metoda 1 - przerobić na operacje na macierzach


test_equality(np.linspace(0,1,1000))

## Ćwiczenie 4
W eksperymencie badano tempo rozwoju larw owadów. Zmierzono wagę pewnej liczby osobników co 24h.

Napisz funkcję `find_most_stable_growth(measurements)`, która:
- przyjmuje macierz 2D pomiarów (wiersze = osobniki, kolumny = punkty czasowe),
- oblicza średnie tempo wzrostu dla każdego osobnika (przyrost wagi na jednostkę czasu),
- zwraca indeks osobnika o najbardziej regularnym wzroście (najmniejsze odchylenie standardowe przyrostów).

*Podpowiedź*: `diff`, `std`, `argmin`

In [ ]:
import numpy as np

# Przykładowe pomiary wagi (g) 5 larw w ciągu 8 dni
measurements = np.array([
    [2.1, 3.5, 5.2, 7.8, 10.1, 13.5, 16.8, 20.2],  # larwa 0
    [1.8, 2.9, 4.1, 5.5, 7.2, 9.1, 11.3, 13.8],    # larwa 1 - najbardziej stabilny wzrost wagi
    [2.3, 4.1, 6.8, 10.2, 14.5, 19.1, 24.3, 30.1], # larwa 2
    [2.0, 3.2, 4.6, 6.3, 8.4, 10.8, 13.5, 16.4],   # larwa 3
    [1.9, 3.0, 4.3, 6.0, 8.1, 10.5, 13.2, 16.1]    # larwa 4
])

def find_most_stable_growth(measurements):
    # trzeba po wszystkich wierszach zebrać informacje dotyczącą średniej - tworzymy array z tymi wynikami
    differences = np.diff(measurements, 1, 1) # n, F
    stds = np.std(differences, 1)
    idx = np.argmin(stds)
    max_val = stds[idx]
    return idx, max_val

most_stable = find_most_stable_growth(measurements)
print(most_stable)  # Oczekiwany wynik: 1

## Ćwiczenie 5

W analizie RNA-seq dostajemy macierz zliczeń (counts), gdzie:
- wiersze odpowiadają genom (np. geny G1...G5000),
- kolumny odpowiadają próbom biologicznym (S1...S10).

1. Wygeneruj sztuczną macierz zliczeń C o wymiarach 5000 x 10, gdzie elementy są liczbami całkowitymi wylosowanymi z rozkładu Poissona (λ=20). (`random`, `poisson`)

2. Jaki jest średni poziom ekspresji każdego genu? Policz średnie wyrażenie każdego genu (średnia po kolumnach) - wektor `mean_expression` długości 5000. (`mean`)

3. Znajdź 10 genów o największej wariancji (wariancja po kolumnach). Zwróć ich indeksy. (`var`, `argsort`)

4. Znormalizuj dane kolumnowo: dla każdej próbki policz sumę zliczeń i podziel wartości w tej kolumnie przez tę sumę (czyli otrzymujemy proporcje zamiast surowych zliczeń).

5. (trudniejsze) Oblicz macierz korelacji Pearsona pomiędzy wszystkimi parami próbek (kolumn) i wskaż parę próbek o największej korelacji. (`corrcoeff`, `unravel_index`, `argmax`, `fill_diagonal`)

In [ ]:
# 1
# tworzenie sztucznej macierzy zliczeń
from numpy import corrcoef


lamb = 20
G = 5000 # ilość genów 
P = 10 # ilość prób
shape = [G, P]
C = np.random.poisson(lamb, shape)

# 2
# średni poziom ekspresji
means = np.mean(C, 0) # ma sens bo jest ~ poissona # (P, )

# 3
# k genów o największej wariancji 
k = 10
gen_expr_var = np.var(C, 1) # (G, )
# to jest argparition - szybciej
gen_expr_var_maxidx_sort = np.argsort(gen_expr_var)[-k:] # (k, )
print(gen_expr_var_maxidx_sort)

# 4
# normalizacja zliczeń 
sum_counts = np.sum(C, 0) # (P, )
C_normalized = C / sum_counts.reshape(1, P) # (G, P)
# testy 
print(C_normalized[:, 1].sum()) # powinna być równa 1 

# 5 
# musimy pomiędzy kolejnymi próbami (po kolumnach)
corr = np.corrcoef(C_normalized, rowvar=False)
np.fill_diagonal(corr, 0)
# para o największej korelacji np.fill_diagonal(corr, 0) # tutaj korelacja = 1
max_idx_flat = np.argmax(corr)
print(max_idx_flat) # daje indeks ale spłaszczony 
max_idx = np.unravel_index(max_idx_flat, corr.shape) # daje indeks zwrócony z powrotem na właściwy wymiar. 
print(max_idx)

# z nich macierz 



# Zadanie *

W tym zadaniu zajmiemy się przykładowym problemem klasyfikacji binarnej. Będziemy rozpatrywać obiekty, które są opisywane przez dwie cechy (zwane też zmiennymi niezależnymi) A i B. Dowolny obiekt określony przez wartości cech A i B należy do dokładnie jednej z dwóch możliwych klas, które będziemy oznaczać $C_I$ i $C_{II}$; inaczej rzecz ujmując, zakładamy, że na przestrzeni cech obiektów zdefiniowany jest pewien podział na dwie rozłączne klasy $C_I$ i $C_{II}$. 

Rozważmy następujący model predykcyjny, który na podstawie danych wartości cechy A ($x^A$) i cechy B ($x^B$) zwraca wartość $M_\mathbf{w}(x^A,x^B)$ interpretowaną jako prawdopodobieństwo przynależności obiektu do klasy $C_{II}$:  

$$M_\mathbf{w}(x^A,x^B) = \frac{1}{1+e^{-(w_Ax^A+w_Bx^B)}},$$

gdzie $\mathbf{w} = (w_A, w_B)$ to odpowiednio wagi dla wartości cech $A$ i $B$. Zwykle wartości wag zapewniające dobre własności predykcyjne modelu znajdowane są w procesie określanym terminem "uczenie pod nadzorem". W tym zadaniu nie będziemy jednak przeprowadzać takiego uczenia. Zamiast tego zbadamy jak nasz model zachowuje się dla pewnych wartości tych wag, tj. dla każdej kombinacji wartości dla $w_A$ i $w_B$, gdzie $w_A \in [0, 0.1, 0.2, ..., 1]$ oraz $w_B \in [2, 2.1, 2.2, ..., 3]$. 

Zachowanie modelu zbadamy na danych $D = \{(x^A_i, x^B_i, \Pr(C_{II})_i)\}_{i=1}^5$ dotyczących pięciu obiektów o wartościach cech i prawdopodobieństwie przynależności do klasy $C_{II}$ przedstawionych w poniższej tabeli:

\begin{array}{|c|c|c|c|}
\hline
\mathbf{i} & \mathbf{x^A_i} & \mathbf{x^B_i} & \mathbf{\Pr(C_{II})_i}\\
\hline
1 & 1.0 & 1.3 & 0\\
\hline
2 & 2.2 & 1.1 & 1\\
\hline
3 & 2.0 & 2.4 & 1\\
\hline
4 & 1.5 & 3.2 & 0\\
\hline
5 & 3.2 & 1.2 & 1\\
\hline
\end{array}

[Tabelka w HTML. Do odkomentowania należy usunąć tę linijkę i znaki \]: # na końcu kodu HTML.
<table>
  <tr>
    <th>A</th>
    <th>B</th>
    <th>$C_{II}</th>
  </tr>
  <tr>
    <th>1.0</th>
    <th>1.3</th>
    <th>0</th>
  </tr>
  <tr>
    <th>2.2</th>
    <th>1.1</th>
    <th>1</th>
  </tr>
  <tr>
    <th>2.0</th>
    <th>2.4</th>
    <th>1</th>
  </tr>
  <tr>
    <th>1.5</th>
    <th>3.2</th>
    <th>0</th>
  </tr>
  <tr>
    <th>3.2</th>
    <th>1.2</th>
    <th>1</th>
  </tr>
</table>]: #
    
Zakładamy, że wiemy do jakiej klasy dany obiekt należy, dlatego w tabeli są tylko dwie wartości prawdopodobieństwa przynależności do klasy $C_{II}$: 1 jeżeli obiekt należy do klasy $C_{II}$ i 0 w przeciwnym przypadku (tzn. obiekt należy do klasy $C_I$). 

Własności predykcyjne naszego modelu dla danej pary wartości wag $w_A$ i $w_B$ ocenimy wyliczając błąd średniokwadratowy (ang. *Mean Squared Error*) dla danych $D$ zadany wzorem

$$MSE_\mathbf{w}(D) = \frac{1}{n}\sum_{i=1}^{n}\left(\Pr(C_{II})_i - M_\mathbf{w}(x^A_i,x^B_i)\right)^2.$$

Proszę napisać program w Pythonie w miarę możliwości wykorzystujący wektoryzację NumPy (tzn. starajmy się jak najmniej korzystać z instrukcji pętli w Pythonie), który oblicza wartości błędu średniokwadratowego dla wszystkich kombinacji wartości wag podanych powyżej.

Wynikiem działania programu powinna być tablica 2D z wartościami błędów dla poszczególnych kombinacji wag.

In [ ]:
# TODO 